Nom Etudiante 1: <font color='green'>Ben Mimoun Sarah</font>
<br>
Nom Etudiante 2: <font color='blue'>Kouidrat Shyrin</font>
<br>
Nom Etudiante 3: <font color='red'>Abunada Dima</font>

In [1]:
# Si vous modifiez votre module motifsSearch.py, les commandes suivantes permettront de le recharger automatiquement les modifications sans devoir redémarrer le notebook.
%load_ext autoreload
%autoreload 2

# TME 2.2 : Projet Detection de motifs - Median String

Les motifs que nous cherchons dans les séquences d'ADN peuvent présenter certaines variations ou mutations. Nous allons développer différents algorithmes pour la recherche de motifs variables (**Partie A**), puis améliorer cette recherche en intégrant une liste étendue de motifs de référence et en développant des fonctions pour l'algorithme Median String (**Partie B**), avant d'appliquer ces approches aux données de ChIP-Seq de _C. glabrata_ (**Partie C**).

## Partie A : Recheche de pattern (motifs) en permettant des variations

1\. Comme dans le TME precedent, nous alons utiliser des données atificielles pour pouvoir tester les algorithmes. Reutilisez les séquences artificielles et le motif de taille `k = 4` sauvegardés le TME précédente.

In [2]:
import readline

with open("seqs_artificielles_sans_motif.txt", "r", encoding="utf-8") as seqs_file :
    seq_artificielles_sans_motif = seqs_file.read()
seq_artificielles_sans_motif = seq_artificielles_sans_motif.splitlines()
print('Sequences artificielles  (sans motif):')
for seq in seq_artificielles_sans_motif: 
    print(seq)


motif_file = "motif_artificiel.txt"
with open(motif_file, "r", encoding="utf-8") as motif_file :
    motif_artificiel = motif_file.read()

print("\nLe motif géneré était :", motif_artificiel) 

Sequences artificielles  (sans motif):
TACCACTTAACGCCGTCAAAATGTGCCTATTTTGGAACGAA
GGATTCTGAAGTGGGAGGGACATTAGTATGCCCATTATTGG
AAACTGACTGTGTATTTCAAAATGCGGGCTCGCGGACTCTT
TCAATCCCCTACAGCCTAATTTTAAGCTGAAACTAGGATAC
CAATGAGGTTGGATTACAGAAAGTTATCCGTTGAACCCCTC
ACGCCGGATATGGCACAGTCGAGGAAAACGAATCTGCAGTA
GTCATTACTATCTGGGCTTTTCTTGCTGCGTCTGAAGTCCT
ATCACCAGCTTTGCGCTTTAGATGGGCTTAACTCATACCAG
TCATATACGGAGGTCGGATTAGTAGGAGACAACATTGTTTA
GTGCGCGCACCTAGTTCTCGGAGAGAACTCCCACAAGGTGG

Le motif géneré était : CTTT


2\. Jusqu'à maintenant, nous avons travaillé avec des motifs invariables. Cependant, dans les données biologiques, les motifs peuvent présenter des variations. Créez une fonction pour faire varier un motif selon un certain nombre de positions `v`.

In [3]:
import random

nucleotides = ('A', 'C', 'G', 'T')

def modifierMotif(motif:str, v:int, upper = True):
    """
    Modifie v positions d'un motif aléatoirement 
    entrée motif: motif à modifier
    entrée v: nombre de positions à varier, si 0 le motif n'est pas modifié
    entrée upper: bool, si True les nucléotides modifiés seront en majuscule, si False en minuscule
    sortie motif_modifie: motif modifié
    """
    i = 0 
    positions = []
    
    while i < v : 
        pos = random.randint(0, len(motif)-1) # on prend une position dans le motif (ne pas prendre le même ?) 
        positions.append(pos)
        i += 1
    modified_motif = ""
    
    j = 0
    for n in motif : 
        if j in positions : 
            nuc = nucleotides[random.randint(0, 3)]
            if (upper) : 
                nuc = nuc.upper()
            else : 
                nuc = nuc.lower()
            modified_motif += nuc
        else : 
            modified_motif += n
        j += 1
    return modified_motif


# Pour la reproductibilité des résultats
seed = 42
random.seed(seed)

v = 1 # nombre de positions variables dans le motif

print("Motif original : ", motif_artificiel)

# Tester la fonction modifierMotif plusieurs fois
for i in range(0, 5):
    print(modifierMotif(motif_artificiel, v, upper = False))

Motif original :  CTTT
aTTT
CTcT
CcTT
aTTT
CTTa


3\. Modifiez la fonction `implantMotifs()` de la séance précédente afin d’ajouter la possibilité de varier le motif implanté. Cette fois-ci, considérez que toutes les séquences contiendront un motif (`f1 = 1`). En revanche, seule une proportion des séquences contenant un motif (`f2 = 0.6`) incluera un motif modifié. Pour faciliter leur identification, implantez les motifs en minuscules.

**BONUS :** Implantez un motif modifié dans une proportion de séquences légèrement inférieure (`f1 = 0.9`). 

In [4]:
def implantMotifsVar(motif:str, sequences:list, v:int, f1 = 1.0, f2 = 0.4, upper = True):
    """
    Insère un motif dans des positions aléatoires des séquences
    entrée motif : motif qui va être implanté dans les séquences
    entrée séquences : liste de séquences
    entrée v: nombre de positions à varier, si 0 le motif sera invariable
    entrée f : fréquence d'implantation, si 1 toutes les séquences contiendront un motif
    entrée f2 : fraction de motifs variables parmi les motifs implantés, si 0 tous les motifs implantés seront identiques
    entrée upper : bool, si False le motif sera en minuscules
    sortie modified_sequences: liste de séquences ayant le motif implanté
    """
    
    modified_sequences = []
    #count_motif_modified = f2 * v * f1 * len(sequences)
    
    if (upper) : 
        motif = motif.upper()
    else : 
        motif = motif.lower()

    to_modify = random.sample(range(len(sequences)), int(f1*len(sequences))) # les indices des qequences qu'on va modifier 
    
    print(to_modify)      
    for i in range (len(sequences)) : 
        if i in to_modify : 
            seq = sequences[i]
            positions = []
            i = 0
            while i < v : 
                positions.append(random.randint(0, len(seq)))
                i += 1
            print(positions)
            modified_seq = ""
            j = 0
            for n in seq : 
                if j in positions : 
                    modified_seq += motif
                else : 
                    modified_seq += n 
                j += 1
            modified_sequences.append(modified_seq)
        else : 
            modified_sequences.append(sequences[i])
    return modified_sequences

f1 = 0.8
f2 = 0.6
v = 2
seqs_artificielles = implantMotifsVar(
    motif_artificiel, seq_artificielles_sans_motif, v, f1, f2, upper = False)    

seqs_artificielles

[0, 1, 3, 8, 4, 5, 9, 2]
[12, 41]
[34, 26]
[14, 28]
[37, 17]
[0, 10]
[27, 21]
[17, 9]
[13, 21]


['TACCACTTAACGctttCGTCAAAATGTGCCTATTTTGGAACGAA',
 'GGATTCTGAAGTGGGAGGGACATTAGctttATGCCCActttTATTGG',
 'AAACTGACTGTGTActttTTCAAAATGCGGGctttTCGCGGACTCTT',
 'TCAATCCCCTACAGCCTctttATTTTAAGCTGAAACTAGGctttTAC',
 'ctttAATGAGGTTctttGATTACAGAAAGTTATCCGTTGAACCCCTC',
 'ACGCCGGATATGGCACAGTCGctttGGAAActttCGAATCTGCAGTA',
 'GTCATTACTATCTGGGCTTTTCTTGCTGCGTCTGAAGTCCT',
 'ATCACCAGCTTTGCGCTTTAGATGGGCTTAACTCATACCAG',
 'TCATATACGctttAGGTCGGctttTTAGTAGGAGACAACATTGTTTA',
 'GTGCGCGCACCTActttTTCTCGGctttGAGAACTCCCACAAGGTGG']

4\. Nous pouvons visualiser les motifs à l'aide des outils de WebLogo : https://weblogo.berkeley.edu/logo.cgi.
Extrayez les motifs implantés dans vos séquences artificielles et visualisez-les à l'aide de WebLogo.
Affichez le logo en utilisant **Markdown**.
(Alternative: https://weblogo.threeplusone.com/create.cgi)

In [5]:
# Extraire les motifs implantés (en majuscules) des séquences artificielles (en minuscules)
motifs_extraits = # Co

print("Motif original : ", motif_artificiel)

for motif_extrait in motifs_extraits:
        print(motif_extrait)

SyntaxError: invalid syntax (2326656801.py, line 2)

![Logo](Nom du fichier logo)

5\. Insérez le motif reverse complémentaire dans d'autres séquences artificielles, et assemblez les deux listes de séquences. Si vous avez réussi le **BONUS** de la question 3, vous pouvez insérer ce motif reverse complémentaire avec une fréquence légèrement plus faible, afin de simuler une présence inégale du motif varié reverse complémentaire.

In [ ]:
# Importez vos fonctions
seqs_artificielles_rv = # Code

print('Ensemble de séquences artificielles (avec motif variable et complémentaire):')
for seq in seqs_artificielles_rv:
    print(seq)

6\. Sauvegardez vos séquences artificielles implantées avec les motifs variables, ainsi que le motif de base, afin de les réutiliser lors des séances suivantes.

In [ ]:
seqs_impl_file = "seqs_artificielles_avec_motif_var.txt"
# Code

## Partie B : Amélioration de la recherche de motifs et développement de l'algorithme Median String 

1\. Précédemment, nous avons recherché des motifs déjà présents dans nos données. Cependant, une meilleur option pourrait être de créer une liste de tous les motifs possibles d'une certaine taille, même si nous ne connaissons pas leur fréquence. Suivant cette logique, combien de motifs (k-mers) de taille `k = 4` pouvons-nous obtenir avec toutes les combinaisons possibles de nucléotides ? Effectuez le calcul avec les fonctions de **Python**.   
**BONUS :** vous pouvez utiliser **Markdown** en combinaison avec [LaTeX](https://fr.wikipedia.org/wiki/LaTeX), un système de composition de documents qui permet de représenter des expressions mathématiques (par example, $E = mc^2$) pour exprimer votre résultat.

In [ ]:
# Calcul en python
k = 4
nb_kmers = len(nucleotides) ** k
print(nb_kmers)

**BONUS :** Résultat exprimé avec LaTeX

<font color='blue'>
La quantité de k-mers possibles de taille 4 est donnée par: $4^{4}$ = 256

    
nb_kmers = $len(nucleotides)^{k}$
</font>

2\. En utilisant les fonctions de la bibliothèque **itertools**, créez tous les k-mers possibles de taille `k = 4`.

In [ ]:
k = 4

from itertools import product
# ?product

# Générer tous les k-mers de taille k ayant de AAA... à TTT...
allkmers_k4 = ["".join(p) for p in product(nucleotides, repeat = k)] # on en assemble k 
# itertools.product sert justement à générer toutes les combinaisons possible

# Visualiser les 10 premiers k-mers
print(allkmers_k4[0:10])

3\. Éliminez les motifs peu complexes pour éviter les calculs inutiles (motifs ayant plusieurs fois une base répétée, comme **AAAAAAC** ou deux bases répétées comme **ACACACAC**). Réutilisez les fonctions déjà définies (`removeLowComplexeHomo()` et `removeLowComplexeHetero()`) de la séance précédente. Faites attention au type d'objet d'entrée pour chaque fonction. Réfléchissez aux meilleurs paramètres pour filtrer les k-mers selon leur taille. Combien de k-mers restent après avoir supprimé ceux qui sont peu complexes ?

**Attention**: votre motif artificiel est peut-être (par hasard) un homopolymère ou hétéropolymère. Attention dans ce cas à ne pas le filtrer. 

In [ ]:
# Importez vos fonctions
# Liste originale
print('Tous les motifs de taille 4 :')
print(len(allkmers_k4))

# Sans homo
# Code
print(f'Sans homopolymères de longueur >= {m}:')
print(len(allkmers_k4_filtre))

# Sans hetero
# Code
print(f'Sans hétéropolymères de longueur >= {n}:')
print(len(allkmers_k4_filtre))

**Bonus**: Utilisez votre fonction `hashTable` du TME précédent et la liste des motifs de longueur 4 sur les nouvelles séquences artificielles, avec motif variable et son complément inverse variable. Votre motif est-il trouvé parmi les top motifs extraits ? À quel point est-il "meilleur" que les autres ? 

In [ ]:
from motifsSearch import hashTable, gatherRevCompMotifs
results_hash_table = # Code
for (i, (motif, count)) in enumerate(results_hash_table.items()):
    if i > 10:
        break
    print(f'Motif: {motif}, Count: {count}')
print(f'Le motif correct est : {motif_artificiel} et son complément inverse {reversecompl(motif_artificiel)}')

4\. Implémentez l'algorithme _"Median String Search"_ pour chercher des motifs de taille variable. Tout d'abord, créez une fonction pour calculer la distance de Hamming.

In [ ]:
def hamDistance(str1:str, str2:str):
    """
    Calcule la distance de Hamming entre deux chaînes de caractères
    entrée str1: chaîne de caractères
    entrée str2: chaîne de caractères
    sortie distance: distance de Hamming
    >>>hamDistance("TTGGTAT", "TTGCTAA")
    2
    """
    
    return distance

hamDistance("TTGGTAT", "TTGCTAA")

5\. Postérieurement, créez une fonction permettant d'obtenir la distance totale minimale entre un motif et un ensemble de séquences.

In [ ]:
# Attention à ne pas comparer des séquences en majuscules avec des minuscules.

def totalDistanceFw(motif:str, sequences):
    """
    Calcul la distance totale d'un motif par rapport à une liste de séquences. 
    entrée motif: motif à comparer, chaîne de caractères
    entrée sequences: liste de séquences
    sortie total_distance: somme des distances de Hamming minimales
    """
    
    return total_distance

totalDistanceFw(motif_artificiel, seqs_artificielles_rv)

6\. Finalment, implémentez l'algorithme complet pour tester une liste de motifs et produire un dictionnaire contenant toutes les distances calculées à partir de tous les motifs testés. 
Trouvez le motif qui donne la distance minimale avec `getTopMotifs()`. Faites attention à ordonner le dictionnaire  dans le bon sens (croissant ou decroissant).
Essayez votre fonction sur vos séquences artificielles, avec ou sans le complément inverse du motif. 

In [ ]:
def medianStringSearchFw(sequences, kmersV):
    """
    Implement l'algorithme MedianStringSearch
    entrée séquences : liste de séquences
    entrée kmersV: Liste de Kmers à chercher
    sortie motif_dist_dict: un dictionnaire contenant les motifs et leurs distances
    """
    
    return motif_dist_dict

In [ ]:
# Importez vos fonctions

print('Sequences artificielles avec motif fw')
top_motifs_median_string = # Code
print(top_motifs_median_string)

print('Sequences artificielles avec motif fw et rv')
top_motifs_median_string = # Code
print(top_motifs_median_string)

print(f'Le motif correct est : {...} et son complément inverse {...}')

Avez-vous trouvé le motif implanté ? Parfois, le motif implanté n'est pas le plus fréquent. Vous pouvez explorer le dictionnaire obtenu avec la fonction  `searchGivenMotif()`.

In [ ]:
# Importez vos fonctions
from motifsSearch import searchGivenMotif
searchGivenMotif(top_motifs_median_string, motif_artificiel)
# Code 

7\. Nos fonction `medianStringSearchFw` et `totalDistanceFw` ne prend pas en compte le fait que le motif peut aussi être présent sous sa forme complémentaire. 
Pour corriger cela, il faut considérer la distance entre une sous séquence et le motif **ou** son complément . Implémentez une nouvelle version `totalDistance` et `medianStringSearch` qui prennent cela en compte. 

In [ ]:
def totalDistance(motif:str, sequences):
    """
    Calcul la totalDistance
    entrée motif: motif à comparer, chaîne de caractères
    entrée sequences: liste de séquences
    sortie total_distance: somme de distance de hamming minimal
    """
    
    return total_distance

totalDistance(motif_artificiel, seqs_artificielles_rv)

In [ ]:
def medianStringSearch(sequences, kmersV):
    """
    Implemente l'algorithme MedianStringSearch
    entrée séquences : liste de séquences
    entrée kmersV: Liste de Kmers à chercher
    sortie motif_dist_dict: un dictionnaire contenant les motifs et leurs distances
    """
    
    # return motif_dist_dict

In [ ]:
print('Sequences artificielles avec motif fw')
top_motifs_median_string = # code
print(top_motifs_median_string)

print('Sequences artificielles avec motif fw et rv')
top_motifs_median_string = # Code
print(top_motifs_median_string)

print(f'Le motif correct est : {...} et son complément inverse {...}')

**Bonus**: La fonction medianStringSearch peut être longue à exécuter pour un grand nombre de kmers et de séquences. Utilisez le module `tqdm` pour affichier une barre de progression. Voir https://tqdm.github.io/.

**BONUS**: Avec cette nouvelle méthode, si un motif a une distance totale `d` aux séquences, quelle sera la distance de son complément inverse aux mêmes séquences ?
Modifiez en conséquence la fonction medianStringSearch pour renvoyer un dictionnaire ayant pour clés `(motif, motif_rv)`, et faisant moins de calculs. 

In [ ]:
# pip install tqdm ou conda install tqdm -y si vous utilisez un environnement conda
from tqdm import tqdm 
# Fonction medianStringSearch modifiée pour afficher une barre de progression et évitant les comparaisons inutiles

## Partie C : Recherche de motifs variable sur vos données

1\. Utilisez le fichier "Sequence_by_Peaks_##.fasta", contenant les régions de peaks identifiées par ChIP-Seq, où se trouve probablement le facteur de transcription recherché. Appliquez l'algorithme Median String pour détecter les motifs Il faut bien evidement enlever les motifs peu complexe. Créez et testez différentes tailles de motifs.  
*Attention*: commencez par des motifs petits pour évaluer le temps de calcul de votre fonction. Après, passez à des motifs plus grands. 

In [ ]:
k = 6 # tester d'autres tailles k = 5, k = 7, etc.
m = 4
n = 2

from motifsSearch import readFasta
peaks = # Code
print("Nombre de séquences dans les peaks : ", len(peaks))


allkmers = # Code

# Liste originel
print(len(allkmers))

allkmers = # Code

# Sans homo
print(len(allkmers))

allkmers = # Code

# Sans hetero
print(len(allkmers))

In [ ]:
peaks_motifs = # Code

peaks_motifs_top = # Code

print("Top motifs :\n", peaks_motifs_top)

2\. Parfois, la TATA box, un type de motif général présent dans les promoteurs de gènes, dépasse le seuil établi pour filtrer les homopolymères et hétéropolymères. Cependant, ces TATA box constituent des motifs généraux plutôt que spécifiques. Une approche "naïve" pour les éliminer consiste à filtrer les motifs contenant une forte proportion de nucléotides T et A. Créez une fonction qui supprime les motifs enrichis en T et A, en veillant à ne les retirer que si les deux nucléotides sont présents.

In [ ]:
def removeTARich(motifs:list, p:int):
    """
    Enlève les motifs contenant > p de T et A en combination
    entrée motifs: liste de motifs
    entrée p: proportion de nucleótides T et A
    sortie motifsClean: liste de motifs sans les motifs TA rich
    """

    return motifsClean
p = 0.7
test_motifs_TA = ['ATTTTCA', 'TAATTTA', 'AAATTAT', 'TCTACGA', 'ACGCATT', 'ACAAGGT', 'GAAATTA', 'TTGTTGT', 'TATAGCA', 'TAGCCTT']

removeTARich(test_motifs_TA, p)

3\. Retestez après avoir enlevé les motifs enrichis en TA.

In [ ]:
p = 0.6
allkmers_lowTA = # Code

peaks_motifs =  # Code

peaks_motifs_top = # Code

print("Top motifs :\n", peaks_motifs_top)

**NOTES IMPORTANTES :** 
Complétez votre module `motifsSearch` avec les nouvelles fonctions de recherche de motifs implémentées. 
Pensez aussi à développer une fonction qui crée de manière automatique une liste de motifs filtrée, prenant en entrée `k`, `m` , `n` et la proportion de `TA`. 

Appliquez les stratégies appris dans ce TME à l'algorithme de la Hash Table :
* Utilisation d'une liste étendue de motifs de référence.
* Suppression des séquences enrichies en T et A.